# Tamreena AI — Agent Exploration Notebook

This notebook explores and validates the full multi-agent pipeline **before** moving to production Python files.

**RAG is hardcoded** in this notebook — the RAG team will wire in Pinecone + embeddings later. Everything else (agents, tools, memory, MongoDB, FastAPI) is real.

### Build order
1. Environment setup & imports  
2. Shared MD memory tools  
3. InBody parser (VLM tool)  
4. Exercise database (MongoDB) + hardcoded RAG stub  
5. Supervisor agent  
6. Exercise Recommender sub-agent  
7. Plan Assembler sub-agent  
8. End-to-end pipeline run  


## Part 1 — Environment Setup & Imports

Load `.env` credentials, initialize the `AzureChatOpenAI` instance that every agent will share, and verify the connection is live before touching anything else.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0.3,
    timeout=60,
    max_retries=2
)

# quick sanity check
resp = llm.invoke("Say 'Tamreena online' and nothing else.")
print(resp.content)

Tamreena online


## Part 2 — Shared MD Memory Tools

All agents communicate through a single markdown file per session.  
- `read_plan_memory(session_id)` — any agent calls this first to get full context  
- `write_plan_memory(session_id, section_title, content)` — append-only, never overwrites  

Sequential execution means no concurrent write conflicts.

In [ ]:
import os, math
from langchain_core.tools import tool

SESSION_DIR = os.path.join("..", "sessions")

@tool
def read_plan_memory(session_id: str) -> str:
    """Read the full shared plan memory file for this session. Call this first before doing any work."""
    path = os.path.join(SESSION_DIR, session_id, "plan.md")
    if not os.path.exists(path):
        return "(plan file not created yet)"
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

@tool
def write_plan_memory(session_id: str, section_title: str, content: str) -> str:
    """Append a completed section to the shared plan memory file. Never call this more than once per section."""
    path = os.path.join(SESSION_DIR, session_id, "plan.md")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"\n\n## {section_title}\n{content}\n\n---")
    return f"Written to plan memory: {section_title}"

@tool
def validate_session_duration(session_id: str) -> str:
    """
    Plan Assembler calls this after scheduling to verify no day exceeds its set budget.
    Parses the assembled Weekly Schedule and DAY MAP from the plan file.
    Returns PASS if all days are within budget, or a violation report with excess counts.
    Plan Assembler must trim and retry if this returns violations.
    """
    path = os.path.join(SESSION_DIR, session_id, "plan.md")
    if not os.path.exists(path):
        return "ERROR: plan file not found"
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    # parse DAY MAP — extract max_sets budget per day label
    day_budgets = {}
    in_day_map = False
    for line in content.splitlines():
        if "DAY MAP:" in line:
            in_day_map = True
            continue
        if in_day_map and line.startswith("Day "):
            if "max_sets:" in line:
                day_label = line.split("—")[0].strip()   # "Day 1"
                max_sets = int(line.split("max_sets:")[1].split("|")[0].strip())
                day_budgets[day_label] = max_sets
        if in_day_map and line.strip() == "":
            in_day_map = False

    # count scheduled sets per day from Weekly Schedule section
    day_set_counts = {}
    current_day = None
    for line in content.splitlines():
        if line.startswith("### Day"):
            current_day = line.split("—")[0].replace("###", "").strip()  # "Day 1"
            day_set_counts[current_day] = 0
        if current_day and "|" in line and "x" in line.lower():
            for part in line.split("|"):
                p = part.strip()
                if "x" in p.lower():
                    try:
                        sets = int(p.lower().split("x")[0].strip())
                        day_set_counts[current_day] = day_set_counts.get(current_day, 0) + sets
                    except ValueError:
                        pass

    violations = []
    for day, scheduled in day_set_counts.items():
        budget = day_budgets.get(day)
        if budget and scheduled > budget:
            excess = scheduled - budget
            violations.append(f"  {day}: {scheduled} sets scheduled, budget {budget} ({excess} over — trim {excess} sets)")

    if not violations:
        return "PASS — all days within session duration budget."
    return "VIOLATIONS — trim before writing final plan:\n" + "\n".join(violations)

# --- test ---
import uuid
TEST_SESSION = str(uuid.uuid4())
print(write_plan_memory.invoke({"session_id": TEST_SESSION, "section_title": "Test", "content": "hello"}))
print(read_plan_memory.invoke({"session_id": TEST_SESSION}))
print("Memory tools + validate_session_duration defined.")

## Part 3 — InBody Parser (VLM Tool)

This is a tool (not a sub-agent) called by the supervisor as its first action.  
It sends the raw InBody scan image/PDF to Azure gpt-4.1-mini with vision and returns:
- Structured fitness data (SMM, BF%, segmental lean mass)
- **Flags** — the most important output, tells sub-agents what to do differently (asymmetries, elevated BF%, etc.)

In this notebook we also define a `parse_inbody_text` fallback for when you pass raw text instead of an image (useful during development).

In [7]:
import base64
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

INBODY_SYSTEM_PROMPT = """You are an InBody scan analysis tool. Extract fitness-relevant data and output structured text.

Output format (follow exactly):

INBODY ANALYSIS
───────────────
Skeletal Muscle Mass : {value} kg
Body Fat %           : {value}%
BMR                  : {value} kcal
Segmental Lean Mass:
  Right arm : {value} kg | Left arm : {value} kg  → Arm asymmetry: YES/NO (diff: {value}g)
  Right leg : {value} kg | Left leg : {value} kg  → Leg asymmetry: YES/NO (diff: {value}g)
  Trunk     : {value} kg

FLAGS (used by all sub-agents)
───────────────────────────────
ARM_ASYMMETRY   : YES/NO  — if YES, pull day must include unilateral movements
LEG_ASYMMETRY   : YES/NO  — if YES, leg day prioritises unilateral, start on weaker side
ELEVATED_BF     : YES/NO  — if YES (>18% male/>25% female), lean toward 12-15 rep ranges
TRUNK_UNDERDEVELOPED : YES/NO — if YES, chest/back volume gets priority

Extract what you can from the scan. If a value is not visible, write UNKNOWN."""


@tool
def parse_inbody(session_id: str, inbody_b64: str, content_type: str) -> str:
    """Parse an InBody scan image or PDF (base64 encoded) and return structured fitness data with flags."""
    msg = HumanMessage(content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:{content_type};base64,{inbody_b64}"},
        },
        {"type": "text", "text": INBODY_SYSTEM_PROMPT},
    ])
    result = llm.invoke([msg])
    return result.content


@tool
def parse_inbody_text(session_id: str, raw_text: str) -> str:
    """Parse InBody data from raw text (use when image is not available during dev/testing)."""
    prompt = f"""{INBODY_SYSTEM_PROMPT}

Raw InBody text to parse:
{raw_text}"""
    result = llm.invoke(prompt)
    return result.content


# --- test with a realistic fake InBody text ---
SAMPLE_INBODY_TEXT = """
InBody 570 Result Sheet
Name: Ahmed Al-Rashidi   Age: 27   Gender: Male

Body Composition Analysis
  Weight: 84.3 kg
  Skeletal Muscle Mass: 38.2 kg
  Body Fat Mass: 18.6 kg
  Body Fat %: 22.1%
  BMR: 1910 kcal

Segmental Lean Analysis (kg)
  Right Arm: 3.82   Left Arm: 3.41
  Right Leg: 10.95  Left Leg: 11.02
  Trunk: 28.7
"""

parsed = parse_inbody_text.invoke({"session_id": "test", "raw_text": SAMPLE_INBODY_TEXT})
print(parsed)

INBODY ANALYSIS
───────────────
Skeletal Muscle Mass : 38.2 kg
Body Fat %           : 22.1%
BMR                  : 1910 kcal
Segmental Lean Mass:
  Right arm : 3.82 kg | Left arm : 3.41 kg  → Arm asymmetry: YES (diff: 410g)
  Right leg : 10.95 kg | Left leg : 11.02 kg  → Leg asymmetry: YES (diff: 70g)
  Trunk     : 28.7 kg

FLAGS (used by all sub-agents)
───────────────────────────────
ARM_ASYMMETRY   : YES  — if YES, pull day must include unilateral movements
LEG_ASYMMETRY   : YES  — if YES, leg day prioritises unilateral, start on weaker side
ELEVATED_BF     : YES  — if YES (>18% male/>25% female), lean toward 12-15 rep ranges
TRUNK_UNDERDEVELOPED : NO — if YES, chest/back volume gets priority


## Part 4 — Exercise Database (MongoDB) + Hardcoded RAG Stub

### 4a — MongoDB exercise tool
Queries the `tamrena.exercises` collection. The `search_exercise_db` tool filters by muscle group, movement type, and contraindications.

### 4b — Hardcoded RAG stub (replaces Pinecone + embeddings)
`search_rag` returns curated, static training principles and exercise notes per muscle group.  
**This is a placeholder** — the RAG team will swap this function for the real Pinecone hybrid search without changing any other code.

In [ ]:
import sqlite3
from langchain_core.tools import tool
from typing import Optional

# ── SQLite exercise database (temporary stand-in until MongoDB is wired up) ──
DB_PATH = os.path.join("..", "data", "tamreena.db")
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

def get_db_connection():
    return sqlite3.connect(DB_PATH)

SEED_EXERCISES = [
    # (name, primary_muscle, movement_type, equipment, difficulty, contraindications)
    ("Flat Barbell Bench Press", "chest", "compound", "barbell", "intermediate", None),
    ("Incline Dumbbell Press", "chest", "compound", "dumbbell", "beginner", None),
    ("Cable Fly", "chest", "isolation", "cable", "beginner", None),
    ("Pec Deck", "chest", "isolation", "machine", "beginner", None),
    ("Weighted Dip", "chest", "compound", "bodyweight", "advanced", "shoulder_pain"),

    ("Pull-Up", "back", "compound", "bodyweight", "intermediate", None),
    ("Lat Pulldown", "back", "compound", "cable", "beginner", None),
    ("Barbell Row", "back", "compound", "barbell", "intermediate", "lower_back_pain"),
    ("Single-Arm Dumbbell Row", "back", "unilateral", "dumbbell", "beginner", None),
    ("Chest-Supported Row", "back", "compound", "machine", "beginner", "lower_back_pain"),
    ("Straight-Arm Pulldown", "back", "isolation", "cable", "beginner", None),

    ("Seated Dumbbell Overhead Press", "shoulders", "compound", "dumbbell", "beginner", None),
    ("Barbell Overhead Press", "shoulders", "compound", "barbell", "intermediate", "shoulder_pain"),
    ("Cable Lateral Raise", "shoulders", "isolation", "cable", "beginner", None),
    ("Dumbbell Lateral Raise", "shoulders", "isolation", "dumbbell", "beginner", None),
    ("Face Pull", "shoulders", "isolation", "cable", "beginner", None),
    ("Reverse Pec Deck", "shoulders", "isolation", "machine", "beginner", None),

    ("Barbell Curl", "arms", "isolation", "barbell", "beginner", None),
    ("Incline Dumbbell Curl", "arms", "isolation", "dumbbell", "beginner", None),
    ("Hammer Curl", "arms", "isolation", "dumbbell", "beginner", None),
    ("Close-Grip Bench Press", "arms", "compound", "barbell", "intermediate", None),
    ("Tricep Pushdown", "arms", "isolation", "cable", "beginner", None),
    ("Single-Arm Cable Curl", "arms", "unilateral", "cable", "beginner", None),

    ("Barbell Back Squat", "legs", "compound", "barbell", "intermediate", "knee_pain"),
    ("Leg Press", "legs", "compound", "machine", "beginner", "knee_pain"),
    ("Romanian Deadlift", "legs", "compound", "barbell", "intermediate", "lower_back_pain"),
    ("Leg Extension", "legs", "isolation", "machine", "beginner", "knee_pain"),
    ("Lying Leg Curl", "legs", "isolation", "machine", "beginner", None),
    ("Bulgarian Split Squat", "legs", "unilateral", "dumbbell", "intermediate", "knee_pain"),
    ("Standing Calf Raise", "legs", "isolation", "machine", "beginner", None),
    ("Hip Thrust", "legs", "compound", "barbell", "intermediate", None),
]

def init_exercise_db():
    conn = get_db_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS exercises (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            primary_muscle TEXT NOT NULL,
            movement_type TEXT NOT NULL,
            equipment TEXT,
            difficulty TEXT,
            contraindications TEXT
        )
    """)
    count = conn.execute("SELECT COUNT(*) FROM exercises").fetchone()[0]
    if count == 0:
        conn.executemany(
            "INSERT INTO exercises (name, primary_muscle, movement_type, equipment, difficulty, contraindications) VALUES (?, ?, ?, ?, ?, ?)",
            SEED_EXERCISES,
        )
        conn.commit()
    conn.close()

init_exercise_db()


@tool
def search_exercise_db(
    muscle_group: str,
    movement_type: str = "all",
    exclude_contraindication: Optional[str] = None,
) -> str:
    """
    Query the exercise database for exercises matching a muscle group.
    movement_type: compound | isolation | unilateral | all
    exclude_contraindication: body part to avoid (e.g. 'knee_pain')
    """
    conn = get_db_connection()
    query = "SELECT name, equipment, difficulty, movement_type, contraindications FROM exercises WHERE primary_muscle = ?"
    params = [muscle_group]
    if movement_type != "all":
        query += " AND movement_type = ?"
        params.append(movement_type)
    rows = conn.execute(query, params).fetchall()
    conn.close()

    if exclude_contraindication:
        rows = [r for r in rows if not (r[4] and exclude_contraindication in r[4])]

    if not rows:
        return f"No exercises found for [{muscle_group}] [{movement_type}]"

    lines = [f"  • {r[0]} ({r[1] or '?'}, {r[2] or '?'}, {r[3] or '?'})" for r in rows]
    return f"DB results — muscle: [{muscle_group}] | type: [{movement_type}]\n" + "\n".join(lines)


# --- test ---
print(search_exercise_db.invoke({"muscle_group": "chest", "movement_type": "compound"}))
print("---")
print(search_exercise_db.invoke({"muscle_group": "back", "movement_type": "unilateral"}))


=== PRINCIPLES ===

[PRINCIPLE] Progressive overload is the primary driver of hypertrophy and strength. Increase load or reps weekly.
[PRINCIPLE] For hypertrophy, train each muscle group 10-20 sets/week across 2+ sessions for optimal frequency.
[PRINCIPLE] Rep ranges: strength 1-5, hypertrophy 6-15, endurance 15+. All ranges build muscle if taken near failure.
[PRINCIPLE] Rest periods: heavy compound 2-3 min, moderate 90s, isolation/corrective 60s.
[PRINCIPLE] RPE (Rate of Perceived Exertion) 8-9 = 1-2 reps in reserve. RPE 5-6 = comfortable, corrective work.
[PRINCIPLE] Elevated BF% (>18% male, >25% female): lean toward higher rep ranges (12-15) for better fat oxidation.
[PRINCIPLE] Asymmetry correction: always start unilateral sets on the weaker side. Never let the stronger side compensate.
[PRINCIPLE] Beginners: 10-12 sets/week per group. Intermediates: 14-18. Advanced: 18-22.

=== CHEST SPECIFIC ===

[CHEST] Chest has two primary functions: horizontal adduction (pressing) and should

## Part 5 — Supervisor Agent

The supervisor is the orchestrator. It:
1. Calls `parse_inbody_text` (or `parse_inbody` for real scans) to extract structured data + flags  
2. Reads the user intake form  
3. Decides the training split based on days/week + experience + goal  
4. Calculates volume per muscle group (with reduction factors for sleep, job type, etc.)  
5. Writes the session header to the shared MD memory file  
6. Dispatches the Exercise Recommender sub-agent once per muscle group (sequential)  
7. Dispatches the Plan Assembler sub-agent  
8. Returns the final synthesised plan to the user  

The supervisor uses `deepagents.create_deep_agent` with `AzureChatOpenAI`.

In [ ]:
from deepagents import create_deep_agent

SUPERVISOR_PROMPT = """You are Tamreena's supervisor agent — the orchestrator of a personalised workout plan generation pipeline.

## Your job
1. Call parse_inbody_text with the raw InBody text provided. Extract the structured data and FLAGS.
2. Based on the user's intake form and InBody data, decide:
   - Training split (Full Body / PPL / Upper-Lower / Body Part based on days_per_week + experience + goal)
   - Which muscle groups to include
   - Intensity per group: hard / medium / soft
   - Weekly volume sets per group (apply reduction factors below)
3. Call write_plan_memory to write the full session header to the shared plan file.
4. Dispatch exercise-recommender for each muscle group IN PRIORITY ORDER (hard first, then medium, then soft).
5. After all muscle groups are done, dispatch plan-assembler.
6. Synthesise and return the final workout plan to the user.

## Split selection rules
- 2 days → Full Body A/B
- 3 days + beginner → Full Body x3
- 3 days + intermediate → Push/Pull/Legs
- 4 days → Upper/Lower x2 (recommended) OR Body Part 4-day for advanced
- 5 days → PPL + Upper + Lower (advanced) OR Body Part 5-day
- 6 days → PPL x2 (A=strength focus, B=hypertrophy focus)

## Intensity assignment rules
Assign intensity based on InBody flags + user's stated priority:
- hard  → muscles the user flagged as priority ("bigger arms", "lagging back") + muscles with InBody asymmetry flags
- medium → all other muscle groups in the plan
- soft  → corrective/stabiliser muscles (rear delts, core, calves)

## DISPATCH ORDER — always hard → medium → soft
The todo list in the plan file MUST be ordered:
  Step 1..N  : all HARD muscle groups first
  Step N+1.. : all MEDIUM muscle groups next
  Step last-1: all SOFT muscle groups last
  Step last  : Assemble weekly schedule

This ordering makes the plan file show what needs the most attention at the top.

## Volume targets per experience level
- beginner: 10-12 sets/week per group
- intermediate: 14-18 sets/week per group
- advanced: 18-22 sets/week per group

## Volume reduction factors
- poor sleep → multiply by 0.8
- heavy physical job → multiply by 0.8
- InBody asymmetry flag on a group → unilateral focus for that group, NOT extra volume

## Intensity prescription rules
- hard  : 4-5 sets × 6-8 reps  | rest 2-3 min | RPE 8-9 | heavy compound first
- medium: 3-4 sets × 10-12 reps | rest 90s     | RPE 7   | compound + isolation
- soft  : 3 sets   × 15+ reps   | rest 60s     | RPE 5-6 | corrective / unilateral

## Elevated BF% rule
If ELEVATED_BF=YES → use the higher end of each rep range (hard=8 reps, medium=12 reps).

## Session header format to write to plan memory (section_title = "Training Plan")
Write exactly this structure so the priority order is visible:

SPLIT: {split_name}
VOLUME TARGET: {sets_per_week} sets/week per group ({experience} level)

PRIORITY ORDER (hard → medium → soft):
[ ] Step 1: {muscle} — HARD    ({N} sets/week) ← highest attention
[ ] Step 2: {muscle} — HARD    ({N} sets/week)
...
[ ] Step X: {muscle} — MEDIUM  ({N} sets/week)
...
[ ] Step Y: {muscle} — SOFT    ({N} sets/week) ← maintenance/corrective
[ ] Step Z: Assemble weekly schedule

ADJUSTMENTS APPLIED:
- {any volume reductions or rep range shifts}
- {any asymmetry unilateral flags}"""

SUPERVISOR_TOOLS = [
    parse_inbody_text,
    parse_inbody,
    read_plan_memory,
    write_plan_memory,
]

def build_supervisor(subagents_list):
    return create_deep_agent(
        model=llm,
        tools=SUPERVISOR_TOOLS,
        subagents=subagents_list,
        system_prompt=SUPERVISOR_PROMPT,
        name="tamreena-supervisor",
    )

print("Supervisor builder defined.")

## Part 6 — Exercise Recommender Sub-agent

One agent definition, called once per muscle group with a different task prompt each time.  
It does **not** have a different identity per muscle — the muscle group and intensity come in via the prompt.

**Process per call:**
1. `read_plan_memory` → gets full InBody data + what previous agents already wrote  
2. `search_rag` → hardcoded stub returns principles + muscle-specific notes  
3. `search_exercise_db` → queries MongoDB for matching movements  
4. Selects 3-5 exercises with full prescription (sets/reps/rest/RPE)  
5. `write_plan_memory` → appends its section to the shared file  
6. Returns the prescription to the supervisor

In [28]:
EXERCISE_RECOMMENDER_PROMPT = """You are Tamreena's exercise recommender. You are called once per muscle group.

## Your process (follow in order)
1. Call read_plan_memory with the session_id to get full context: InBody analysis, flags, training plan, and any previous muscle group prescriptions.
2. Call search_rag with the muscle_group and a query describing what you need (e.g. "hypertrophy chest compound movements").
3. Call search_exercise_db with the muscle_group. If the InBody flags an asymmetry for this group, also call with movement_type="unilateral".
4. Select 3-5 exercises based on RAG guidance and DB results. Choose exercises appropriate for the intensity level.
5. Write the full prescription using write_plan_memory.
6. Return the prescription summary to the supervisor.

## Intensity prescription rules
| Intensity | Sets  | Reps  | Rest    | RPE  | Focus                        |
|-----------|-------|-------|---------|------|------------------------------|
| hard      | 4-5   | 6-8   | 2-3 min | 8-9  | heavy compound first         |
| medium    | 3-4   | 10-12 | 90s     | 7    | compound + isolation         |
| soft      | 3     | 15+   | 60s     | 5-6  | corrective / unilateral      |

## Asymmetry rule
If InBody flagged an asymmetry for THIS muscle group → at least one exercise MUST be unilateral. Always note "start on weaker side."

## Elevated BF% rule
If ELEVATED_BF=YES → use the higher end of the rep range (hard=8, medium=12).

## Output format for write_plan_memory (section_title = "{MUSCLE_GROUP} — {INTENSITY}")
```
1. Exercise Name   {sets}×{reps} | Rest {time} | RPE {n}
   → why this exercise / key technique cue
2. ...
Evidence: [1 sentence from RAG supporting this selection]
```"""

EXERCISE_RECOMMENDER = {
    "model": llm,
    "tools": [read_plan_memory, write_plan_memory, search_rag, search_exercise_db],
    "system_prompt": EXERCISE_RECOMMENDER_PROMPT,
    "name": "exercise-recommender",
    "description": "Recommends 3-5 exercises with full prescription (sets/reps/rest/RPE) for a single muscle group, using RAG guidance and the exercise DB."
}

print("Exercise Recommender sub-agent defined.")

Exercise Recommender sub-agent defined.


## Part 7 — Plan Assembler Sub-agent

Called once, after all muscle group prescriptions are written to the shared memory file.  
It reads everything the exercise recommender agents wrote and arranges it into a coherent weekly schedule.

**Rules it applies:**
- Never train the same muscle group on consecutive days  
- Hardest session placed where recovery time is longest after it  
- Upper/Lower splits alternate upper and lower days  
- Each session gets a warm-up note specific to its muscle group focus  
- Calculates total weekly volume per muscle group and flags if under/over target

In [29]:
PLAN_ASSEMBLER_PROMPT = """You are Tamreena's plan assembler. You are called once after all muscle group prescriptions are complete.

## Your process (follow in order)
1. Call read_plan_memory with the session_id to get all muscle group prescriptions and the split type.
2. Arrange the prescriptions into a weekly schedule following the rules below.
3. Call write_plan_memory with section_title="Weekly Schedule" to save the final plan.
4. Return the full formatted weekly plan to the supervisor.

## Scheduling rules (mandatory)
- NEVER place the same muscle group on consecutive days.
- Place the hardest session (most volume / heaviest loading) where the longest recovery window follows it (e.g. before a rest day).
- For Upper/Lower splits: alternate Upper → Lower → Upper → Lower.
- For PPL: Push → Pull → Legs in order, repeat if 6 days.

## Session format
For each training day output:

### Day {N} — {DayName}: {Session Focus}
**Warm-up:** {2-sentence specific warm-up for this session's muscle focus}

| # | Exercise | Sets × Reps | Rest | RPE |
|---|----------|-------------|------|-----|
| 1 | ...      | ...         | ...  | ... |

**Coaching notes:** {1 key tip for this session}

---

## End of plan output
After all days, add:

### Weekly Volume Summary
| Muscle Group | Sets/Week | Target | Status |
|---|---|---|---|
| chest | X | 14-18 | ✓ / ⚠ under / ⚠ over |
...

### Recovery Notes
- {any asymmetry corrections to remind the user of}
- {any BF% or sleep-based adjustments made}"""

PLAN_ASSEMBLER = {
    "model": llm,
    "tools": [read_plan_memory, write_plan_memory],
    "system_prompt": PLAN_ASSEMBLER_PROMPT,
    "name": "plan-assembler",
    "description": "Arranges all completed muscle-group prescriptions into a full weekly training schedule, following split and recovery rules."
}

print("Plan Assembler sub-agent defined.")

Plan Assembler sub-agent defined.


## Part 8 — End-to-End Pipeline Run

Wire everything together and run the full pipeline with a realistic sample user.  
After this cell runs you can open `sessions/{session_id}/plan.md` to inspect the shared memory file and see every agent's output in order.

In [ ]:
import uuid
from datetime import datetime

# ── Sample user intake ─────────────────────────────────────────────────────────
USER_INTAKE = """USER INTAKE FORM
─────────────────────────────────────
Goal             : hypertrophy
Days per week    : 4
Experience       : intermediate
Session duration : 75min
Injuries/limits  : none
Priority focus   : bigger arms, lagging back
Age              : 27
Sleep quality    : average
Job type         : desk"""

SAMPLE_INBODY = """
InBody 570 Result Sheet
Name: Ahmed Al-Rashidi   Age: 27   Gender: Male

Body Composition Analysis
  Weight: 84.3 kg
  Skeletal Muscle Mass: 38.2 kg
  Body Fat Mass: 18.6 kg
  Body Fat %: 22.1%
  BMR: 1910 kcal

Segmental Lean Analysis (kg)
  Right Arm: 3.82   Left Arm: 3.41
  Right Leg: 10.95  Left Leg: 11.02
  Trunk: 28.7
"""

# ── Build pipeline ─────────────────────────────────────────────────────────────
SESSION_ID = str(uuid.uuid4())
print(f"Session: {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

# ── Run ────────────────────────────────────────────────────────────────────────
user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke({
    "messages": [{"role": "user", "content": user_message}]
})

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

Session: edad9acb-1838-4e0e-9e1d-b70aa5713509

Starting pipeline at 20:27:17...



In [ ]:
# Inspect the shared memory file — shows every agent's output in order
session_file = os.path.join("..", "sessions", SESSION_ID, "plan.md")
if os.path.exists(session_file):
    with open(session_file, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Session file not found — pipeline may not have written to memory yet.")

In [ ]:
result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)